In [1]:

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import mlflow
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

MLFLOW_EXPERIMENT = "dengue_log_tweedie_experiments"
DATA_PATH = "../data/processed/dengue_features_advanced_sprint.parquet"


/media/breezy/NewVolume/projects_int/dengue/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

df = pd.read_parquet(DATA_PATH)
df = df.sort_values(["district", "week_start"]).reset_index(drop=True)

# Separate Train/Val (2007 - 2023) and Test (2024 - 2026)
train_val_df = df[df["year"] <= 2023].copy()
test_df = df[df["year"] >= 2024].copy()

# Remove missing targets from training pool
train_val_df = train_val_df.query("cases.notna()").copy()
test_df = test_df.query("cases.notna()").copy()

# Setup matrices
drop_cols = ["district", "week_start", "cases", "is_missing_observation", "weather_complete", "weather_missing_days"]
X_train_val = train_val_df.drop(columns=drop_cols)
y_train_val = train_val_df["cases"].astype(np.float32)

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["cases"].astype(np.float32)

print("X_train_val shape:", X_train_val.shape)
print("X_test shape:", X_test.shape)


X_train_val shape: (22776, 61)
X_test shape: (3406, 61)


In [3]:

cv = TimeSeriesSplit(n_splits=5)

def eval_metrics(y_true, y_pred):
    # Cap predictions to avoid negative cases or infinite values
    y_pred = np.clip(y_pred, 0, np.inf)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    outbreak_thr = np.quantile(y_true, 0.95)
    out_mask = y_true >= outbreak_thr
    out_mae = mean_absolute_error(y_true[out_mask], y_pred[out_mask]) if out_mask.sum() > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Outbreak_MAE": out_mae}

# MLflow setup
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment(MLFLOW_EXPERIMENT)


2026/08/21 11:24:01 INFO mlflow.tracking.fluent: Experiment with name 'dengue_log_tweedie_experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='/media/breezy/NewVolume/projects_int/dengue/notebooks/../mlruns/402924219552658261', creation_time=1787291641119, effective_trace_archival_retention=None, experiment_id='402924219552658261', last_update_time=1787291641119, lifecycle_stage='active', name='dengue_log_tweedie_experiments', tags={}, trace_location=None, workspace='default'>

In [4]:

def objective_log1p_lgb(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 256),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1,
        'n_estimators': 500
    }
    
    cv_maes = []
    
    # Target is log1p transformed
    y_train_val_log = np.log1p(y_train_val)
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_val)):
        X_tr, y_tr_log = X_train_val.iloc[train_idx], y_train_val_log.iloc[train_idx]
        X_va, y_va_log = X_train_val.iloc[val_idx], y_train_val_log.iloc[val_idx]
        y_va_raw = y_train_val.iloc[val_idx].values
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr_log)
        
        preds_log = model.predict(X_va)
        preds_raw = np.expm1(preds_log) # Inverse transform
        
        fold_mae = mean_absolute_error(y_va_raw, preds_raw)
        cv_maes.append(fold_mae)
        
    return np.mean(cv_maes)

print("Starting Optuna Study for Log1p Transformation...")
study_log1p = optuna.create_study(direction='minimize')
study_log1p.optimize(objective_log1p_lgb, n_trials=30)
print("Best Params Log1p:", study_log1p.best_params)


[I 2026-08-21 11:24:01,127] A new study created in memory with name: no-name-7dd0fafc-e08a-4476-9d68-ffaa8de67ba0


Starting Optuna Study for Log1p Transformation...


[I 2026-08-21 11:24:32,097] Trial 0 finished with value: 13.80494151624595 and parameters: {'learning_rate': 0.005885953069638533, 'num_leaves': 193, 'max_depth': 11, 'min_child_samples': 36, 'subsample': 0.6393737904943988, 'colsample_bytree': 0.9336482856880804, 'reg_alpha': 7.93625776981747e-07, 'reg_lambda': 0.006090982365681439}. Best is trial 0 with value: 13.80494151624595.


[I 2026-08-21 11:24:36,632] Trial 1 finished with value: 13.54819548844581 and parameters: {'learning_rate': 0.04238100396728327, 'num_leaves': 32, 'max_depth': 10, 'min_child_samples': 100, 'subsample': 0.9166263336927678, 'colsample_bytree': 0.8856916320958494, 'reg_alpha': 0.21785795731845642, 'reg_lambda': 1.3409120737245332e-07}. Best is trial 1 with value: 13.54819548844581.


[I 2026-08-21 11:24:41,710] Trial 2 finished with value: 20.149897771949817 and parameters: {'learning_rate': 0.0025487352816785246, 'num_leaves': 242, 'max_depth': 8, 'min_child_samples': 100, 'subsample': 0.9618808072022864, 'colsample_bytree': 0.682120753961658, 'reg_alpha': 6.062714378081886, 'reg_lambda': 5.656186568924488e-07}. Best is trial 1 with value: 13.54819548844581.


...

[I 2026-08-21 11:27:17,989] Trial 26 finished with value: 13.007396514797694 and parameters: {'learning_rate': 0.010597362818264726, 'num_leaves': 97, 'max_depth': 7, 'min_child_samples': 12, 'subsample': 0.8026207334567621, 'colsample_bytree': 0.6484780913175416, 'reg_alpha': 1.3204000354135865, 'reg_lambda': 1.542652050768175e-05}. Best is trial 19 with value: 12.967735312924148.


[I 2026-08-21 11:27:33,553] Trial 27 finished with value: 13.126359752111 and parameters: {'learning_rate': 0.008890418403547347, 'num_leaves': 142, 'max_depth': 9, 'min_child_samples': 10, 'subsample': 0.7168376738948079, 'colsample_bytree': 0.6554251463710995, 'reg_alpha': 1.8463174863723524, 'reg_lambda': 7.637923002756181e-07}. Best is trial 19 with value: 12.967735312924148.


[I 2026-08-21 11:27:43,591] Trial 28 finished with value: 14.472457386607427 and parameters: {'learning_rate': 0.0050263480783738265, 'num_leaves': 102, 'max_depth': 7, 'min_child_samples': 26, 'subsample': 0.8128098186629409, 'colsample_bytree': 0.7188662741765971, 'reg_alpha': 0.08725559821467938, 'reg_lambda': 9.527234649867422e-08}. Best is trial 19 with value: 12.967735312924148.


[I 2026-08-21 11:27:45,844] Trial 29 finished with value: 13.466161406312645 and parameters: {'learning_rate': 0.011505945112122867, 'num_leaves': 183, 'max_depth': 4, 'min_child_samples': 16, 'subsample': 0.9452759421491186, 'colsample_bytree': 0.6069115821865012, 'reg_alpha': 8.140867112139807, 'reg_lambda': 9.247299649184453e-06}. Best is trial 19 with value: 12.967735312924148.


Best Params Log1p: {'learning_rate': 0.030247166840052204, 'num_leaves': 60, 'max_depth': 6, 'min_child_samples': 22, 'subsample': 0.9168535801306819, 'colsample_bytree': 0.5671789608639416, 'reg_alpha': 1.0519202224118114, 'reg_lambda': 2.5042793214330003e-05}


In [5]:

def objective_tweedie_lgb(trial):
    params = {
        'objective': 'tweedie',
        'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.05, 1.95),
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 256),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1,
        'n_estimators': 500
    }
    
    cv_maes = []
    
    # Target is RAW (Tweedie applies internal log link)
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_val)):
        X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
        X_va, y_va = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict(X_va)
        fold_mae = mean_absolute_error(y_va, preds)
        cv_maes.append(fold_mae)
        
    return np.mean(cv_maes)

print("Starting Optuna Study for Tweedie Objective...")
study_tweedie = optuna.create_study(direction='minimize')
study_tweedie.optimize(objective_tweedie_lgb, n_trials=30)
print("Best Params Tweedie:", study_tweedie.best_params)


[I 2026-08-21 11:27:45,850] A new study created in memory with name: no-name-3a9666d9-2e08-4b92-aa28-de4256d59c50


Starting Optuna Study for Tweedie Objective...


[I 2026-08-21 11:27:47,384] Trial 0 finished with value: 17.708171426252502 and parameters: {'tweedie_variance_power': 1.9198690474163138, 'learning_rate': 0.0034964794532122262, 'num_leaves': 152, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7430576524955538, 'colsample_bytree': 0.7335530309742616, 'reg_alpha': 0.0024999293124461745, 'reg_lambda': 3.4925467659592373e-07}. Best is trial 0 with value: 17.708171426252502.


[I 2026-08-21 11:28:09,468] Trial 1 finished with value: 26.21305539458142 and parameters: {'tweedie_variance_power': 1.516610833014005, 'learning_rate': 0.0011937978087040865, 'num_leaves': 229, 'max_depth': 15, 'min_child_samples': 43, 'subsample': 0.7730695976267649, 'colsample_bytree': 0.9716549441324027, 'reg_alpha': 6.991351934301742e-05, 'reg_lambda': 3.130973879338816e-05}. Best is trial 0 with value: 17.708171426252502.


[I 2026-08-21 11:28:20,172] Trial 2 finished with value: 13.190654417632214 and parameters: {'tweedie_variance_power': 1.7235053485541734, 'learning_rate': 0.008199044055709313, 'num_leaves': 130, 'max_depth': 11, 'min_child_samples': 69, 'subsample': 0.9403191506106621, 'colsample_bytree': 0.5641931896555923, 'reg_alpha': 0.003183604623921775, 'reg_lambda': 0.0002256359094884973}. Best is trial 2 with value: 13.190654417632214.


...

[I 2026-08-21 11:31:38,893] Trial 26 finished with value: 12.889194516452454 and parameters: {'tweedie_variance_power': 1.625130278750505, 'learning_rate': 0.01898463238844901, 'num_leaves': 101, 'max_depth': 12, 'min_child_samples': 25, 'subsample': 0.9113231087479741, 'colsample_bytree': 0.9602629495904984, 'reg_alpha': 0.0007362575914669239, 'reg_lambda': 5.8377929784453494e-08}. Best is trial 17 with value: 12.746062619789914.


[I 2026-08-21 11:31:53,828] Trial 27 finished with value: 13.806324604094593 and parameters: {'tweedie_variance_power': 1.4758927693265356, 'learning_rate': 0.005450823457335914, 'num_leaves': 135, 'max_depth': 14, 'min_child_samples': 58, 'subsample': 0.7945792762495754, 'colsample_bytree': 0.8082515089508895, 'reg_alpha': 2.0723230734097543e-05, 'reg_lambda': 2.908506553938573e-07}. Best is trial 17 with value: 12.746062619789914.


[I 2026-08-21 11:32:02,084] Trial 28 finished with value: 19.45583703150752 and parameters: {'tweedie_variance_power': 1.7152552085134438, 'learning_rate': 0.002363159215192029, 'num_leaves': 53, 'max_depth': 11, 'min_child_samples': 38, 'subsample': 0.7070552536345333, 'colsample_bytree': 0.7612436768196487, 'reg_alpha': 1.5687767650951367, 'reg_lambda': 3.8097075394757365e-06}. Best is trial 17 with value: 12.746062619789914.


[I 2026-08-21 11:32:19,656] Trial 29 finished with value: 12.826065749170297 and parameters: {'tweedie_variance_power': 1.3827127293103993, 'learning_rate': 0.010966795753749483, 'num_leaves': 139, 'max_depth': 15, 'min_child_samples': 19, 'subsample': 0.8501485940193515, 'colsample_bytree': 0.502742713133013, 'reg_alpha': 0.013056659066201135, 'reg_lambda': 1.1253949559206099e-08}. Best is trial 17 with value: 12.746062619789914.


Best Params Tweedie: {'tweedie_variance_power': 1.5693916817798006, 'learning_rate': 0.01092173075279431, 'num_leaves': 114, 'max_depth': 13, 'min_child_samples': 59, 'subsample': 0.8733227825719496, 'colsample_bytree': 0.6480723603092854, 'reg_alpha': 0.00011845663777379248, 'reg_lambda': 1.3418707933458363e-08}


In [6]:

def evaluate_and_log(strategy_name, params, use_log1p=False):
    params['n_estimators'] = 500
    params['random_state'] = 42
    
    # Test on Holdout
    if use_log1p:
        params['objective'] = 'regression'
        y_tr_log = np.log1p(y_train_val)
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train_val, y_tr_log)
        
        preds_test_log = model.predict(X_test)
        preds_test = np.expm1(preds_test_log)
    else:
        params['objective'] = 'tweedie'
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train_val, y_train_val)
        preds_test = model.predict(X_test)
        
    test_metrics = eval_metrics(y_test.values, preds_test)
    print(f"--- {strategy_name} Test Results ---")
    print(test_metrics)
    
    # Evaluate Fold 3 explicitly (The 2017 Epidemic)
    # Re-run CV to get fold 3 metrics
    fold3_metrics = {}
    
    with mlflow.start_run(run_name=f"{strategy_name}_Final"):
        mlflow.log_params(params)
        for k, v in test_metrics.items():
            mlflow.log_metric(f"Test_{k}", v)
            
        for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_val)):
            X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
            X_va, y_va = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]
            
            if use_log1p:
                model.fit(X_tr, np.log1p(y_tr))
                preds_val = np.expm1(model.predict(X_va))
            else:
                model.fit(X_tr, y_tr)
                preds_val = model.predict(X_va)
                
            metrics = eval_metrics(y_va.values, preds_val)
            mlflow.log_metric(f"Fold_{fold+1}_MAE", metrics["MAE"])
            mlflow.log_metric(f"Fold_{fold+1}_RMSE", metrics["RMSE"])
            
            if fold == 2: # Fold 3
                fold3_metrics = metrics
                
        print(f"--- {strategy_name} Fold 3 (2017 Outbreak) Results ---")
        print(fold3_metrics)

evaluate_and_log("LightGBM_Log1p", study_log1p.best_params, use_log1p=True)
evaluate_and_log("LightGBM_Tweedie", study_tweedie.best_params, use_log1p=False)


--- LightGBM_Log1p Test Results ---
{'MAE': 14.900595843553633, 'RMSE': np.float64(52.223223318972806), 'R2': 0.6258420504940078, 'Outbreak_MAE': 107.04289153160326}


--- LightGBM_Log1p Fold 3 (2017 Outbreak) Results ---
{'MAE': 6.119650323904814, 'RMSE': np.float64(23.458515660484785), 'R2': 0.6956558926843801, 'Outbreak_MAE': 44.73914988204698}


--- LightGBM_Tweedie Test Results ---
{'MAE': 13.737632245499158, 'RMSE': np.float64(48.119152441214965), 'R2': 0.6823392422171928, 'Outbreak_MAE': 94.34453490409035}


--- LightGBM_Tweedie Fold 3 (2017 Outbreak) Results ---
{'MAE': 6.384477880473676, 'RMSE': np.float64(22.81911772023641), 'R2': 0.7120205235803707, 'Outbreak_MAE': 41.38446462602413}
